# 03 — Chronological train/validation/test split

Only date coverage, volume, and label prevalence are inspected here. A random stratified split would preserve class rates but would let later orders inform predictions for earlier orders. The broad continuous timeline supports the operationally realistic chronological alternative.

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert ROOT.name == "02-olist-late-delivery-ml", f"Run from assignment root or notebooks/, got {ROOT}"
SEED = 42
pd.set_option("display.max_columns", 100)
df=pd.read_parquet(ROOT/"artifacts/02_labeled/labeled_orders.parquet").sort_values(["order_purchase_timestamp","order_id"]).reset_index(drop=True)
timeline=df.assign(month=df.order_purchase_timestamp.dt.to_period("M").astype(str)).groupby("month").late.agg(["size","mean"])
{"range":(df.order_purchase_timestamp.min(),df.order_purchase_timestamp.max()),"months":len(timeline),"timeline":timeline}

{'range': (Timestamp('2016-09-15 12:16:38'), Timestamp('2018-08-29 15:00:37')),
 'months': 23,
 'timeline':          size      mean
 month                  
 2016-09     1  1.000000
 2016-10   270  0.007407
 2016-12     1  0.000000
 2017-01   750  0.029333
 2017-02  1653  0.029643
 2017-03  2546  0.045562
 2017-04  2303  0.065567
 2017-05  3545  0.029901
 2017-06  3135  0.030303
 2017-07  3872  0.027893
 2017-08  4193  0.029096
 2017-09  4150  0.043855
 2017-10  4478  0.041760
 2017-11  7288  0.124040
 2017-12  5513  0.074551
 2018-01  7069  0.057009
 2018-02  6556  0.141397
 2018-03  7003  0.189633
 2018-04  6798  0.045013
 2018-05  6749  0.065639
 2018-06  6096  0.011647
 2018-07  6156  0.033788
 2018-08  6351  0.061880}

In [2]:
n=len(df); i=int(n*.70); j=int(n*.85)
train=df.iloc[:i].copy(); validation=df.iloc[i:j].copy(); test=df.iloc[j:].copy()
assert len(train)+len(validation)+len(test)==n
assert train.order_purchase_timestamp.max() <= validation.order_purchase_timestamp.min() <= validation.order_purchase_timestamp.max() <= test.order_purchase_timestamp.min()
assert set(train.order_id).isdisjoint(validation.order_id) and set(train.order_id).isdisjoint(test.order_id) and set(validation.order_id).isdisjoint(test.order_id)
split_summary={k:{"rows":len(v),"pct":len(v)/n*100,"start":str(v.order_purchase_timestamp.min()),"end":str(v.order_purchase_timestamp.max()),"late_count":int(v.late.sum()),"late_rate":float(v.late.mean())} for k,v in {"train":train,"validation":validation,"test":test}.items()}
split_summary

{'train': {'rows': 67533,
  'pct': 69.99979269455616,
  'start': '2016-09-15 12:16:38',
  'end': '2018-04-15 20:07:56',
  'late_count': 5291,
  'late_rate': 0.07834688226496676},
 'validation': {'rows': 14471,
  'pct': 14.999585389112319,
  'start': '2018-04-15 20:10:23',
  'end': '2018-06-21 07:50:39',
  'late_count': 624,
  'late_rate': 0.04312072420703476},
 'test': {'rows': 14472,
  'pct': 15.000621916331522,
  'start': '2018-06-21 08:29:29',
  'end': '2018-08-29 15:00:37',
  'late_count': 620,
  'late_rate': 0.042841348811498065}}

In [3]:
out=ROOT/"artifacts/03_splits"; out.mkdir(parents=True,exist_ok=True)
for name,part in {"train":train,"validation":validation,"test":test}.items(): part.to_parquet(out/f"{name}.parquet",index=False)
(out/"split_summary.json").write_text(json.dumps({"strategy":"chronological 70/15/15, sorted by purchase timestamp then order_id","splits":split_summary},indent=2))
assert all((out/f"{x}.parquet").exists() for x in ["train","validation","test"])
split_summary

{'train': {'rows': 67533,
  'pct': 69.99979269455616,
  'start': '2016-09-15 12:16:38',
  'end': '2018-04-15 20:07:56',
  'late_count': 5291,
  'late_rate': 0.07834688226496676},
 'validation': {'rows': 14471,
  'pct': 14.999585389112319,
  'start': '2018-04-15 20:10:23',
  'end': '2018-06-21 07:50:39',
  'late_count': 624,
  'late_rate': 0.04312072420703476},
 'test': {'rows': 14472,
  'pct': 15.000621916331522,
  'start': '2018-06-21 08:29:29',
  'end': '2018-08-29 15:00:37',
  'late_count': 620,
  'late_rate': 0.042841348811498065}}